In [43]:
import CAST
import scanpy as sc
import os
import numpy as np
import warnings
import dgl
import torch
import CAST
import os
import numpy as np
import anndata as ad
import scanpy as sc
import warnings
import pandas as pd
warnings.filterwarnings("ignore")
import scanpy as sc
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [12]:
indexMerge = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/alignedIndexMerge.csv")
print(indexMerge.head())

   SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60
0        46608         49890        100805        147668        198498
1        46607         49891        100806        147669        198499
2        46606         49893        100808        147671        198501
3        46605         49894        100809        147672        198502
4        46604         49895        100810        147673        198503


In [3]:
indexMerge5000 = indexMerge.head(5000)
print(indexMerge5000.tail())

      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60
4995        40378         57220        106599        155142        205744
4996        40377         57221        106600        155143        205745
4997        40376         57222        106601        155144        205746
4998        40375         57223        106602        155145        205747
4999        40374         57224        106603        155146        205748


In [13]:
coordsLabelIntensity = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_calculateByClusterLR/coordsLabelIntensity.csv")
print(coordsLabelIntensity.head())

   Unnamed: 0  sample  cell_label   57.0346   58.0296   59.0139   67.0191  \
0           1       0           5  0.208880  0.254289  0.499496  0.399597   
1           2       0           5  0.000000  0.223221  0.480784  0.257563   
2           3       0           5  0.000000  0.000000  1.175326  0.720719   
3           4       0           5  0.152048  0.000000  0.557510  0.000000   
4           5       0           5  0.000000  0.000000  0.725190  0.382195   

    71.0137   71.0503   72.0091  ...  856.5052  859.5297   860.536  863.5618  \
0  1.607469  0.127144  0.962665  ...  0.363270  0.481332  0.163471  0.000000   
1  2.481191  0.000000  1.210547  ...  0.712591  0.154538  0.283319  0.394930   
2  0.698543  0.000000  0.155232  ...  0.177408  0.155232  0.221760  0.410255   
3  0.726453  0.000000  0.278755  ...  0.084471  0.000000  0.194284  0.304097   
4  2.587164  0.000000  0.000000  ...  0.000000  0.000000  0.000000  0.088199   

   864.5684  882.5853  921.5991   923.615  985.6046  986

In [5]:
print(coordsLabelIntensity.columns)

Index(['Unnamed: 0', 'sample', 'cell_label', '57.0346', '58.0296', '59.0139',
       '67.0191', '71.0137', '71.0503', '72.0091',
       ...
       '856.5052', '859.5297', '860.536', '863.5618', '864.5684', '882.5853',
       '921.5991', '923.615', '985.6046', '986.6105'],
      dtype='object', length=699)


In [6]:
import pandas as pd

# 创建一个新的空列，用来存储 cell_label 的值
indexMerge5000['cell_label'] = None

# 遍历 indexMerge5000 的每一行
for index, row in indexMerge5000.iterrows():
    # 遍历每一行的前5列（假设前5列是 SpotIndex）
    for i in range(5):
        SpotIndex = row[i]  # 获取当前的 SpotIndex 值
        # 获取对应 SpotIndex 的 cell_label 值
        cellLabel = coordsLabelIntensity[coordsLabelIntensity.iloc[:, 0] == SpotIndex]['cell_label']
        
        # 如果找到了符合条件的 cell_label，填充到对应位置
        if not cellLabel.empty:
            # 获取第一个匹配的 cell_label（如果有多个匹配）
            indexMerge5000.at[index, f'cell_label_{i+1}'] = cellLabel.iloc[0]

# 返回新的数据框
print(indexMerge5000)
         
        

      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
0           46608         49890        100805        147668        198498   
1           46607         49891        100806        147669        198499   
2           46606         49893        100808        147671        198501   
3           46605         49894        100809        147672        198502   
4           46604         49895        100810        147673        198503   
...           ...           ...           ...           ...           ...   
4995        40378         57220        106599        155142        205744   
4996        40377         57221        106600        155143        205745   
4997        40376         57222        106601        155144        205746   
4998        40375         57223        106602        155145        205747   
4999        40374         57224        106603        155146        205748   

     cell_label  cell_label_1  cell_label_2  cell_label_3  cell_label_4  \


In [7]:
indexMerge5000.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabel.csv")

In [8]:
# 假设后5列是 'cell_label_1', 'cell_label_2', 'cell_label_3', 'cell_label_4', 'cell_label_5'
# 选取最后5列
last_5_columns = indexMerge5000.iloc[:, -5:]

# 计算每行中不同值的个数
nunique_counts = last_5_columns.apply(lambda row: row.nunique(), axis=1)

# 统计每行不同值个数对应的数量
count_5_equal = (nunique_counts == 1).sum()  # 同行内5个值全相等
count_4_equal = (nunique_counts == 2).sum()  # 同行内4个值全相等
count_3_equal = (nunique_counts == 3).sum()  # 同行内3个值全相等
count_2_equal = (nunique_counts == 4).sum()  # 同行内2个值全相等
count_1_equal = (nunique_counts == 5).sum()  # 同行内1个值全相等

# 输出统计结果
print(f"同行内5个值全相等的行数: {count_5_equal}")
print(f"同行内4个值全相等的行数: {count_4_equal}")
print(f"同行内3个值全相等的行数: {count_3_equal}")
print(f"同行内2个值全相等的行数: {count_2_equal}")
print(f"同行内1个值全相等的行数: {count_1_equal}")

同行内5个值全相等的行数: 177
同行内4个值全相等的行数: 1586
同行内3个值全相等的行数: 2346
同行内2个值全相等的行数: 839
同行内1个值全相等的行数: 52


In [16]:
indexMerge5000=pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabel.csv")

In [18]:
indexMerge5000 = indexMerge5000.iloc[:,1:]

In [19]:
indexMerge5000

,SpotIndex_0,SpotIndex_15,SpotIndex_30,SpotIndex_45,SpotIndex_60,cell_label,cell_label_1,cell_label_2,cell_label_3,cell_label_4,cell_label_5
0,46608,49890,100805,147668,198498,NaN,14.0,14.0,5.0,14.0,15.0
1,46607,49891,100806,147669,198499,NaN,14.0,14.0,5.0,14.0,15.0
2,46606,49893,100808,147671,198501,NaN,14.0,14.0,5.0,14.0,15.0
3,46605,49894,100809,147672,198502,NaN,14.0,15.0,5.0,14.0,15.0
4,46604,49895,100810,147673,198503,NaN,14.0,0.0,5.0,14.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
4995,40378,57220,106599,155142,205744,NaN,0.0,7.0,8.0,15.0,19.0
4996,40377,57221,106600,155143,205745,NaN,11.0,7.0,8.0,15.0,19.0
4997,40376,57222,106601,155144,205746,NaN,11.0,8.0,19.0,15.0,19.0
4998,40375,57223,106602,155145,205747,NaN,11.0,8.0,19.0,19.0,19.0


In [25]:
print(coordsLabelIntensity.columns[2:])

Index(['cell_label', '57.0346', '58.0296', '59.0139', '67.0191', '71.0137',
       '71.0503', '72.0091', '72.017', '73.0294',
       ...
       '856.5052', '859.5297', '860.536', '863.5618', '864.5684', '882.5853',
       '921.5991', '923.615', '985.6046', '986.6105'],
      dtype='object', length=697)


In [ ]:
print(coordsLabelIntensity.columns[3:])

Index(['57.0346', '58.0296', '59.0139', '67.0191', '71.0137', '71.0503',
       '72.0091', '72.017', '73.0294', '74.0246',
       ...
       '856.5052', '859.5297', '860.536', '863.5618', '864.5684', '882.5853',
       '921.5991', '923.615', '985.6046', '986.6105'],
      dtype='object', length=696)


In [21]:
columns_from_third = coordsLabelIntensity.columns[2:]
for col_name in columns_from_third:
    for index, row in indexMerge5000.iterrows():
        for i in range(5):
            SpotIndex = row[i]
            intensity = coordsLabelIntensity[coordsLabelIntensity.iloc[:, 0] == SpotIndex][col_name]
            indexMerge5000.at[index, f'{col_name}_intensity_{i+1}'] = intensity.iloc[0]
print(indexMerge5000)


      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
0           46608         49890        100805        147668        198498   
1           46607         49891        100806        147669        198499   
2           46606         49893        100808        147671        198501   
3           46605         49894        100809        147672        198502   
4           46604         49895        100810        147673        198503   
...           ...           ...           ...           ...           ...   
4995        40378         57220        106599        155142        205744   
4996        40377         57221        106600        155143        205745   
4997        40376         57222        106601        155144        205746   
4998        40375         57223        106602        155145        205747   
4999        40374         57224        106603        155146        205748   

      cell_label  cell_label_1  cell_label_2  cell_label_3  cell_label_4  .

In [22]:
indexMerge5000.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabelIntensity.csv")

In [23]:
# Step 1: 删除前六列
indexMerge5000 = indexMerge5000.iloc[:, 6:]

# Step 2: 根据第七列内容进行分组
grouped = indexMerge5000.groupby(indexMerge5000.columns[0])
print(indexMerge5000.columns[0])

# Step 3: 计算每组的均值
grouped_mean = grouped.mean()

# Step 4: 生成新的 DataFrame（包括分组组别和均值）
result = pd.DataFrame(grouped_mean)

# 在结果中加入分组列（分组组别就是索引）
result['Group'] = result.index

# Step 5: 打印结果
print(result)



cell_label_1
              cell_label_2  cell_label_3  cell_label_4  cell_label_5  \
cell_label_1                                                           
0.0               8.783658      9.161560      9.522748     10.461467   
3.0               6.700000      6.714286      6.071429      5.571429   
4.0               6.010081      8.763105      6.711694      7.034274   
7.0               6.561983      9.041322      8.107438      7.595041   
8.0               7.314642      9.171340      8.919003      8.862928   
11.0              7.963074      7.248503      7.979042      9.006986   
12.0              9.750000      7.250000      4.625000      4.625000   
14.0             10.689427     11.030837     10.671806     10.755507   
15.0             10.920299      9.657534     10.494396     10.745953   
19.0              9.315789      8.000000      7.552632      7.973684   

              cell_label_intensity_1  cell_label_intensity_2  \
cell_label_1                                              

In [31]:
result = result.iloc[:,9:-1]

In [44]:
resultT=result.T

In [46]:
resultT

cell_label_1,0.0,3.0,4.0,7.0,8.0,11.0,12.0,14.0,15.0,19.0
57.0346_intensity_1,0.722039,0.729203,0.703077,0.796478,0.767732,0.699072,0.415506,0.780066,0.648102,0.725446
57.0346_intensity_2,0.730046,0.593861,0.676084,0.796848,0.741010,0.698226,0.822039,0.732632,0.704557,0.739008
57.0346_intensity_3,0.772447,0.815812,0.831542,0.890423,0.796846,0.791504,0.826578,0.757683,0.764891,0.757713
57.0346_intensity_4,0.837151,0.865827,0.833392,0.796445,0.735629,0.834324,0.624160,0.747145,0.865541,0.774811
57.0346_intensity_5,0.836345,0.889281,0.894153,0.882823,0.923006,0.844212,0.783603,0.836092,0.885174,0.955123
...,...,...,...,...,...,...,...,...,...,...
986.6105_intensity_1,1.139940,1.042290,1.110942,1.165263,1.085988,1.122400,1.603398,1.146755,1.124966,1.024360
986.6105_intensity_2,1.278079,1.365796,1.293756,1.291239,1.325183,1.298674,1.212408,1.249817,1.301753,1.254971
986.6105_intensity_3,1.582053,1.623641,1.492990,1.476615,1.392002,1.536696,1.154898,1.787234,1.714024,1.556051
986.6105_intensity_4,1.543759,1.343172,1.526142,1.495283,1.644278,1.579068,1.674956,1.569674,1.521299,1.612790


In [45]:
# 假设 'result' 是一个 DataFrame，按照每5行分组
# 提取每5行的组名（从每组的第一行的行名中提取组名）
group_names = []
for i in range(0, len(resultT), 5):
    group_name = resultT.index[i].split('_')[0]  # 获取每组的组名（行名的第一个部分）
    group_names.append(group_name)

# 将数据重新组织为按组名分组的 DataFrame
grouped_result = pd.DataFrame()

for i, group_name in enumerate(group_names):
    # 提取每5行的数据
    group_data = resultT.iloc[i*5:(i+1)*5]
    # 将每组的行合并为一个新 DataFrame，使用组名作为索引
    group_data.index = [group_name] * len(group_data)
    grouped_result = pd.concat([grouped_result, group_data])

# 输出结果
print(grouped_result)

cell_label_1      0.0       3.0       4.0       7.0       8.0       11.0  \
57.0346       0.722039  0.729203  0.703077  0.796478  0.767732  0.699072   
57.0346       0.730046  0.593861  0.676084  0.796848  0.741010  0.698226   
57.0346       0.772447  0.815812  0.831542  0.890423  0.796846  0.791504   
57.0346       0.837151  0.865827  0.833392  0.796445  0.735629  0.834324   
57.0346       0.836345  0.889281  0.894153  0.882823  0.923006  0.844212   
...                ...       ...       ...       ...       ...       ...   
986.6105      1.139940  1.042290  1.110942  1.165263  1.085988  1.122400   
986.6105      1.278079  1.365796  1.293756  1.291239  1.325183  1.298674   
986.6105      1.582053  1.623641  1.492990  1.476615  1.392002  1.536696   
986.6105      1.543759  1.343172  1.526142  1.495283  1.644278  1.579068   
986.6105      1.532903  1.586362  1.485797  1.664443  1.607579  1.544764   

cell_label_1      12.0      14.0      15.0      19.0  
57.0346       0.415506  0.780066

In [47]:
grouped_result

cell_label_1,0.0,3.0,4.0,7.0,8.0,11.0,12.0,14.0,15.0,19.0
57.0346,0.722039,0.729203,0.703077,0.796478,0.767732,0.699072,0.415506,0.780066,0.648102,0.725446
57.0346,0.730046,0.593861,0.676084,0.796848,0.741010,0.698226,0.822039,0.732632,0.704557,0.739008
57.0346,0.772447,0.815812,0.831542,0.890423,0.796846,0.791504,0.826578,0.757683,0.764891,0.757713
57.0346,0.837151,0.865827,0.833392,0.796445,0.735629,0.834324,0.624160,0.747145,0.865541,0.774811
57.0346,0.836345,0.889281,0.894153,0.882823,0.923006,0.844212,0.783603,0.836092,0.885174,0.955123
...,...,...,...,...,...,...,...,...,...,...
986.6105,1.139940,1.042290,1.110942,1.165263,1.085988,1.122400,1.603398,1.146755,1.124966,1.024360
986.6105,1.278079,1.365796,1.293756,1.291239,1.325183,1.298674,1.212408,1.249817,1.301753,1.254971
986.6105,1.582053,1.623641,1.492990,1.476615,1.392002,1.536696,1.154898,1.787234,1.714024,1.556051
986.6105,1.543759,1.343172,1.526142,1.495283,1.644278,1.579068,1.674956,1.569674,1.521299,1.612790


In [49]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 假设 'grouped_result' 是之前的 DataFrame，按照索引列分组
x_values = np.array([0, 15, 30, 45, 60]).reshape(-1, 1)  # x 轴

# 初始化线性回归模型
model = LinearRegression()

# 创建一个空的列表，用来存储每个组的回归结果
regression_results = []

# 按索引分组
grouped = grouped_result.groupby(grouped_result.index)

# Step 1: 对每个组进行回归计算
for group_name, group_data in grouped:
    group_results = {}  # 存储每个组的回归结果
    
    # 对组内每列数据进行回归
    for col_name in group_data.columns:
        y_values = group_data[col_name].values  # y值
        
        # 进行线性回归
        model.fit(x_values, y_values)
        
        # 获取回归方程参数
        slope = model.coef_[0]
        intercept = model.intercept_
        r2 = model.score(x_values, y_values)  # 计算 R² 值
        
        # 将结果存储到字典中，列名格式为原列名加上后缀 '_*'
        group_results[f'{col_name}_Slope'] = slope
        group_results[f'{col_name}_Intercept'] = intercept
        group_results[f'{col_name}_R2'] = r2
    
    # 将每个组的回归结果存入列表
    regression_results.append(group_results)

# Step 2: 使用 pd.DataFrame() 将结果列表转换为 DataFrame
regression_df = pd.DataFrame(regression_results)

# 将回归结果的行名设置为原来的组名
regression_df.index = grouped_result.index.unique()

# 输出回归结果
print(regression_df)

# 如果需要保存结果
regression_df.to_csv("regression_results.csv")

          0.0_Slope  0.0_Intercept    0.0_R2  3.0_Slope  3.0_Intercept  \
57.0346    0.027191       0.957364  0.899274   0.031190       0.882459   
58.0296    0.076526      10.862330  0.927994   0.071546      10.952201   
59.0139    0.005696       0.969281  0.662329   0.005294       0.972598   
67.0191    0.022844       1.425931  0.924611   0.027365       1.061195   
71.0137    0.004832       0.354641  0.939345   0.002511       0.403849   
...             ...            ...       ...        ...            ...   
882.5853   0.010121       1.288729  0.736867   0.007241       1.325251   
921.5991   0.007011       1.205026  0.723780   0.007103       1.179148   
923.615   -0.042867      27.488463  0.785317  -0.069703      29.294080   
985.6046   0.006528       2.136533  0.640848   0.011085       2.083646   
986.6105   0.014553       4.801845  0.491922   0.021447       4.359945   

            3.0_R2  4.0_Slope  4.0_Intercept    4.0_R2  7.0_Slope  ...  \
57.0346   0.983046   0.026609       0

In [50]:
regression_df

,0.0_Slope,0.0_Intercept,0.0_R2,3.0_Slope,3.0_Intercept,3.0_R2,4.0_Slope,4.0_Intercept,4.0_R2,7.0_Slope,...,12.0_R2,14.0_Slope,14.0_Intercept,14.0_R2,15.0_Slope,15.0_Intercept,15.0_R2,19.0_Slope,19.0_Intercept,19.0_R2
57.0346,0.027191,0.957364,0.899274,0.031190,0.882459,0.983046,0.026609,0.878310,0.922289,0.026980,...,0.948539,0.026231,1.075242,0.796514,0.029048,1.016968,0.840368,0.025334,0.952080,0.903385
58.0296,0.076526,10.862330,0.927994,0.071546,10.952201,0.845568,0.077768,10.559663,0.862189,0.073594,...,0.834553,0.071965,11.354711,0.870142,0.080309,10.774528,0.889374,0.082710,10.312537,0.932800
59.0139,0.005696,0.969281,0.662329,0.005294,0.972598,0.738219,0.004303,1.014072,0.566417,0.002708,...,0.623991,0.005281,0.982170,0.632900,0.004609,0.979503,0.559094,0.006945,0.884176,0.869829
67.0191,0.022844,1.425931,0.924611,0.027365,1.061195,0.991022,0.021640,1.330748,0.913155,0.023336,...,0.325513,0.024859,1.493823,0.780915,0.023498,1.532033,0.819848,0.018278,1.513390,0.829727
71.0137,0.004832,0.354641,0.939345,0.002511,0.403849,0.256757,0.004682,0.315247,0.965984,0.001648,...,0.002720,0.004893,0.349594,0.621730,0.004710,0.337187,0.802730,0.005124,0.303817,0.993336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882.5853,0.010121,1.288729,0.736867,0.007241,1.325251,0.676905,0.011063,1.206324,0.849840,0.010089,...,0.369171,0.010707,1.360609,0.545915,0.010908,1.353752,0.571029,0.013055,1.170506,0.748253
921.5991,0.007011,1.205026,0.723780,0.007103,1.179148,0.523673,0.006547,1.185506,0.770815,0.008016,...,0.273773,0.007228,1.239646,0.441717,0.008222,1.212110,0.647019,0.010394,1.102821,0.864023
923.615,-0.042867,27.488463,0.785317,-0.069703,29.294080,0.688217,-0.034755,27.083010,0.525699,-0.060022,...,0.183486,-0.055522,28.018939,0.656564,-0.034451,27.047125,0.567605,-0.023374,26.826112,0.247259
985.6046,0.006528,2.136533,0.640848,0.011085,2.083646,0.748870,0.007356,2.103834,0.869305,0.007585,...,0.784031,0.007589,2.099998,0.627159,0.006172,2.100488,0.634209,0.009731,1.952000,0.842918


In [54]:
regression_df.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/MeanByCellLabel1LRResults.csv",index=False)

In [53]:
# 假设 'regression_df' 已经是计算好的回归结果

# Step 1: 提取所有包含 R2 的列
r2_columns = [col for col in regression_df.columns if '_R2' in col]

# Step 2: 创建一个字典来存储每个条件下符合的行数
r2_conditions = {0.9: 0, 0.8: 0, 0.7: 0}

# Step 3: 对每一行（代谢物）进行统计
for idx, row in regression_df.iterrows():
    # 计算当前代谢物符合各个条件的簇数
    r2_values = row[r2_columns]
    
    # 计算符合条件的簇数
    for threshold in r2_conditions.keys():
        # 统计 R2 大于等于 threshold 的细胞簇数量
        count = (r2_values >= threshold).sum()
        if count >= len(r2_values) / 2:
        # if count == len(r2_values):  
            r2_conditions[threshold] += 1

# Step 4: 输出结果
print(r2_conditions)

{0.9: 245, 0.8: 436, 0.7: 519}


↑大于等于半数label中R2大于等于对应值的代谢物个数

In [2]:
cellLabel = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabel.csv")

In [3]:
cellLabel

,Unnamed: 0,SpotIndex_0,SpotIndex_15,SpotIndex_30,SpotIndex_45,SpotIndex_60,cell_label,cell_label_1,cell_label_2,cell_label_3,cell_label_4,cell_label_5
0,0,46608,49890,100805,147668,198498,NaN,14.0,14.0,5.0,14.0,15.0
1,1,46607,49891,100806,147669,198499,NaN,14.0,14.0,5.0,14.0,15.0
2,2,46606,49893,100808,147671,198501,NaN,14.0,14.0,5.0,14.0,15.0
3,3,46605,49894,100809,147672,198502,NaN,14.0,15.0,5.0,14.0,15.0
4,4,46604,49895,100810,147673,198503,NaN,14.0,0.0,5.0,14.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,40378,57220,106599,155142,205744,NaN,0.0,7.0,8.0,15.0,19.0
4996,4996,40377,57221,106600,155143,205745,NaN,11.0,7.0,8.0,15.0,19.0
4997,4997,40376,57222,106601,155144,205746,NaN,11.0,8.0,19.0,15.0,19.0
4998,4998,40375,57223,106602,155145,205747,NaN,11.0,8.0,19.0,19.0,19.0


In [5]:
label = cellLabel.iloc[:,-5:]

In [6]:
label

,cell_label_1,cell_label_2,cell_label_3,cell_label_4,cell_label_5
0,14.0,14.0,5.0,14.0,15.0
1,14.0,14.0,5.0,14.0,15.0
2,14.0,14.0,5.0,14.0,15.0
3,14.0,15.0,5.0,14.0,15.0
4,14.0,0.0,5.0,14.0,0.0
...,...,...,...,...,...
4995,0.0,7.0,8.0,15.0,19.0
4996,11.0,7.0,8.0,15.0,19.0
4997,11.0,8.0,19.0,15.0,19.0
4998,11.0,8.0,19.0,19.0,19.0


In [7]:
label = label.sort_values(by = label.columns.tolist(),ascending=True).reset_index(drop = True)

In [8]:
label

,cell_label_1,cell_label_2,cell_label_3,cell_label_4,cell_label_5
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
4995,19.0,19.0,14.0,0.0,15.0
4996,19.0,19.0,14.0,15.0,0.0
4997,19.0,19.0,14.0,15.0,0.0
4998,19.0,19.0,14.0,15.0,14.0


In [9]:
label.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabelSorted.csv")

In [10]:
values = label.values
maxCounts = [np.unique(row,return_counts=True)[1].max() for row in values]
sameSum = pd.Series(maxCounts).value_counts().sort_index(ascending=False)

54321个值相同的行数

In [11]:
sameSum

5     177
4     860
3    2031
2    1880
1      52
Name: count, dtype: int64

In [14]:
for index, row in indexMerge.iterrows():
    for i in range(5):
        SpotIndex = row[i]
        cellLabel = coordsLabelIntensity[coordsLabelIntensity.iloc[:, 0] == SpotIndex]['cell_label']
        indexMerge.at[index, f'cell_label_{i+1}'] = cellLabel.iloc[0]

print(indexMerge)

       SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
0            46608         49890        100805        147668        198498   
1            46607         49891        100806        147669        198499   
2            46606         49893        100808        147671        198501   
3            46605         49894        100809        147672        198502   
4            46604         49895        100810        147673        198503   
...            ...           ...           ...           ...           ...   
37931          849        100545        145640        196444        247057   
37932          770        100602        145690        196484        247110   
37933          863        100531        145691        196485        247111   
37934          862        100532        145692        196486        247112   
37935          343        100804        145695        196489        247115   

       cell_label_1  cell_label_2  cell_label_3  cell_label_4  

In [16]:
label1 = indexMerge.iloc[:,-5:]

In [17]:
label1

,cell_label_1,cell_label_2,cell_label_3,cell_label_4,cell_label_5
0,14.0,14.0,5.0,14.0,15.0
1,14.0,14.0,5.0,14.0,15.0
2,14.0,14.0,5.0,14.0,15.0
3,14.0,15.0,5.0,14.0,15.0
4,14.0,0.0,5.0,14.0,0.0
...,...,...,...,...,...
37931,4.0,14.0,15.0,15.0,15.0
37932,4.0,11.0,11.0,0.0,15.0
37933,11.0,11.0,11.0,0.0,15.0
37934,0.0,11.0,11.0,0.0,15.0


In [18]:
values = label1.values
maxCounts = [np.unique(row,return_counts=True)[1].max() for row in values]
sameSum = pd.Series(maxCounts).value_counts().sort_index(ascending=False)
print(sameSum)

5     4184
4     8412
3    15285
2     9839
1      216
Name: count, dtype: int64


In [ ]:
#👆所有对齐点链中，相同值的个数以及有该个数相同值的行数

In [ ]:
#选ROI分簇拟合标准曲线基本流程👇

In [21]:
coordsLabelIntensity = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_calculateByClusterLR/coordsLabelIntensity.csv")
print(coordsLabelIntensity.head())

   Unnamed: 0  sample  cell_label   57.0346   58.0296   59.0139   67.0191  \
0           1       0           5  0.208880  0.254289  0.499496  0.399597   
1           2       0           5  0.000000  0.223221  0.480784  0.257563   
2           3       0           5  0.000000  0.000000  1.175326  0.720719   
3           4       0           5  0.152048  0.000000  0.557510  0.000000   
4           5       0           5  0.000000  0.000000  0.725190  0.382195   

    71.0137   71.0503   72.0091  ...  856.5052  859.5297   860.536  863.5618  \
0  1.607469  0.127144  0.962665  ...  0.363270  0.481332  0.163471  0.000000   
1  2.481191  0.000000  1.210547  ...  0.712591  0.154538  0.283319  0.394930   
2  0.698543  0.000000  0.155232  ...  0.177408  0.155232  0.221760  0.410255   
3  0.726453  0.000000  0.278755  ...  0.084471  0.000000  0.194284  0.304097   
4  2.587164  0.000000  0.000000  ...  0.000000  0.000000  0.000000  0.088199   

   864.5684  882.5853  921.5991   923.615  985.6046  986

In [23]:
coordsLabelIntensity

,Unnamed: 0,sample,cell_label,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
0,1,0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
1,2,0,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
2,3,0,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
3,4,0,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
4,5,0,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247356,247357,60,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
247357,247358,60,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
247358,247359,60,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
247359,247360,60,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [24]:
indexMergeSlice = indexMerge.iloc[5000:10000,:]
print(indexMergeSlice.tail())


      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
9995        34992         62906        112532        161362        211751   
9996        34991         62907        112533        161363        211752   
9997        34990         62908        112534        161364        211753   
9998        34989         62909        112535        161365        211754   
9999        34988         62910        112536        161366        211755   

      cell_label_1  cell_label_2  cell_label_3  cell_label_4  cell_label_5  
9995           0.0           0.0          11.0          11.0          11.0  
9996          19.0           0.0          11.0           4.0           8.0  
9997           8.0           0.0           4.0           4.0           8.0  
9998           7.0           0.0           8.0           8.0           7.0  
9999           7.0          19.0           8.0           8.0           7.0  


In [25]:
for index, row in indexMergeSlice.iterrows():
    for i in range(5):
        SpotIndex = row[i]
        cellLabel = coordsLabelIntensity[coordsLabelIntensity.iloc[:, 0] == SpotIndex]['cell_label']
        indexMergeSlice.at[index, f'cell_label_{i+1}'] = cellLabel.iloc[0]

print(indexMergeSlice)

      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
5000        40373         57225        106604        155147        205749   
5001        40371         57227        106605        155148        205750   
5002        40370         57228        106606        155149        205751   
5003        40369         57229        106607        155150        205752   
5004        40180         57424        106608        155151        205753   
...           ...           ...           ...           ...           ...   
9995        34992         62906        112532        161362        211751   
9996        34991         62907        112533        161363        211752   
9997        34990         62908        112534        161364        211753   
9998        34989         62909        112535        161365        211754   
9999        34988         62910        112536        161366        211755   

      cell_label_1  cell_label_2  cell_label_3  cell_label_4  cell_label_5 

In [26]:
print(coordsLabelIntensity.columns[3:])

Index(['57.0346', '58.0296', '59.0139', '67.0191', '71.0137', '71.0503',
       '72.0091', '72.017', '73.0294', '74.0246',
       ...
       '856.5052', '859.5297', '860.536', '863.5618', '864.5684', '882.5853',
       '921.5991', '923.615', '985.6046', '986.6105'],
      dtype='object', length=696)


In [27]:
intensity_dict = coordsLabelIntensity.set_index(
    coordsLabelIntensity.columns[0]
).to_dict(orient='index')
columns_from_third = coordsLabelIntensity.columns[3:]

for index, row in indexMergeSlice.iterrows():
    for i in range(5):
        SpotIndex = row[i]
        if SpotIndex in intensity_dict:
            for col_name in columns_from_third:
                indexMergeSlice.at[
                    index,
                    f'{col_name}_intensity_{i+1}'
                ] = intensity_dict[SpotIndex].get(col_name, None)

print(indexMergeSlice)

      SpotIndex_0  SpotIndex_15  SpotIndex_30  SpotIndex_45  SpotIndex_60  \
5000        40373         57225        106604        155147        205749   
5001        40371         57227        106605        155148        205750   
5002        40370         57228        106606        155149        205751   
5003        40369         57229        106607        155150        205752   
5004        40180         57424        106608        155151        205753   
...           ...           ...           ...           ...           ...   
9995        34992         62906        112532        161362        211751   
9996        34991         62907        112533        161363        211752   
9997        34990         62908        112534        161364        211753   
9998        34989         62909        112535        161365        211754   
9999        34988         62910        112536        161366        211755   

      cell_label_1  cell_label_2  cell_label_3  cell_label_4  cell_label_5 

In [28]:
indexMergeSlice.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabelIntensity10000.csv")

In [30]:
indexMergeSlice = pd.concat([indexMergeSlice.iloc[:,[5]],indexMergeSlice.iloc[:,10:]],axis=1)

In [31]:
indexMergeSlice

,cell_label_1,57.0346_intensity_1,58.0296_intensity_1,59.0139_intensity_1,67.0191_intensity_1,71.0137_intensity_1,71.0503_intensity_1,72.0091_intensity_1,72.017_intensity_1,73.0294_intensity_1,...,856.5052_intensity_5,859.5297_intensity_5,860.536_intensity_5,863.5618_intensity_5,864.5684_intensity_5,882.5853_intensity_5,921.5991_intensity_5,923.615_intensity_5,985.6046_intensity_5,986.6105_intensity_5
5000,11.0,1.168489,0.000000,7.716435,2.976339,18.122598,2.006273,2.050367,0.727550,12.346296,...,7.727948,7.206619,4.017307,3.557310,1.257325,3.311978,1.839988,2.606649,2.146652,1.379991
5001,4.0,0.000000,0.986138,4.677112,0.000000,19.271955,1.605996,2.394907,0.000000,0.000000,...,4.000034,4.455734,4.708900,3.417750,3.544334,1.645583,0.000000,2.354450,3.848134,2.379767
5002,8.0,0.000000,0.947775,3.257976,2.754470,18.422371,0.000000,1.421662,0.000000,7.937613,...,3.824863,7.649726,6.365665,6.502267,7.075996,3.087211,2.650084,2.103675,2.677404,0.000000
5003,8.0,0.520458,0.718728,4.684121,1.239186,17.869055,0.000000,0.223053,0.991348,5.303714,...,4.497955,8.358140,6.646232,6.847633,3.155282,1.946876,1.913309,0.000000,1.711908,0.872738
5004,7.0,1.221654,1.284303,0.783112,1.691521,22.428321,0.000000,3.069798,1.534899,3.696288,...,4.640416,6.134787,5.715315,8.101065,2.543053,1.861410,4.194726,1.651674,1.966278,1.205984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.0,2.003301,0.000000,5.574402,1.676676,20.228982,1.132300,1.110525,1.350050,3.440451,...,4.847940,5.748272,4.951825,5.609759,0.000000,2.666367,2.043060,2.874136,1.108101,0.657935
9996,19.0,0.921801,1.021456,3.238762,2.989626,27.529477,1.395159,5.630463,0.000000,5.256760,...,2.883837,8.342528,6.007994,3.639127,2.986831,1.510581,1.338924,0.000000,0.961279,0.000000
9997,8.0,1.329249,0.000000,5.150841,0.000000,23.400327,0.000000,1.163093,0.747703,6.230856,...,4.264201,9.459664,7.131508,6.616863,5.048422,1.666469,2.499704,1.838018,2.034073,2.009566
9998,7.0,0.000000,0.000000,3.789200,2.052483,16.135675,0.000000,1.420950,3.157666,9.346692,...,3.704502,7.924822,4.970598,4.994044,0.000000,1.359881,2.227391,1.570897,2.016375,0.562709


In [33]:
grouped = indexMergeSlice.groupby(indexMergeSlice.columns[0])

print(indexMergeSlice.columns[0])

grouped_mean = grouped.mean()

result = pd.DataFrame(grouped_mean)

cell_label_1


In [34]:
result

,57.0346_intensity_1,58.0296_intensity_1,59.0139_intensity_1,67.0191_intensity_1,71.0137_intensity_1,71.0503_intensity_1,72.0091_intensity_1,72.017_intensity_1,73.0294_intensity_1,74.0246_intensity_1,...,856.5052_intensity_5,859.5297_intensity_5,860.536_intensity_5,863.5618_intensity_5,864.5684_intensity_5,882.5853_intensity_5,921.5991_intensity_5,923.615_intensity_5,985.6046_intensity_5,986.6105_intensity_5
cell_label_1,,,,,,,,,,,,,,,,,,,,,
0.0,0.722421,1.185498,4.867079,1.163283,20.821434,0.505552,1.733283,0.650330,6.755551,4.055000,...,4.803597,8.434923,5.305360,5.426705,3.186375,2.102046,2.023530,2.425084,1.840423,1.553651
1.0,0.822678,1.173516,4.845421,1.213751,20.810679,0.651254,1.731944,0.590218,6.772300,4.396370,...,4.899375,8.601300,5.995407,6.905861,4.403831,2.298298,2.028652,2.204691,1.780322,1.641532
3.0,0.697872,1.110374,4.742928,1.275674,20.659190,0.607214,1.740649,0.726549,6.673749,4.432330,...,4.914281,8.563332,5.757006,6.493864,3.961762,2.299211,2.031387,2.334966,1.858622,1.567569
4.0,0.693764,1.103839,4.420415,1.225371,19.898902,0.491434,1.666815,0.674502,6.712750,3.825050,...,4.855595,8.415435,5.519532,5.237804,3.225677,2.190763,1.981652,2.435808,1.798131,1.540583
5.0,0.890684,1.650921,2.738265,0.508819,13.279997,0.227555,3.030463,0.754470,3.383475,3.627631,...,4.632186,7.662429,5.569092,5.044555,3.836829,1.785888,1.881029,2.629958,2.271891,2.144233
6.0,0.833798,1.202757,4.959094,1.339695,21.823991,0.503563,1.827047,0.686701,6.852514,4.183523,...,4.916804,8.819565,6.215196,7.665293,4.515404,2.169479,2.017307,2.003079,1.910919,1.645229
7.0,0.677206,1.184140,4.687989,1.268536,21.109624,0.554591,1.732268,0.743003,7.036537,4.408139,...,4.869278,8.314519,5.741728,6.035845,3.623742,2.257390,2.057763,2.317324,1.849382,1.570597
8.0,0.754835,1.165626,4.580690,1.200813,20.372920,0.531263,1.715115,0.651037,6.946123,4.264466,...,5.038941,8.345983,5.574924,5.714601,3.508755,2.258639,2.025759,2.346867,1.890134,1.560787
11.0,0.698106,1.174150,4.794491,1.233202,20.373712,0.510345,1.678110,0.616679,6.694550,3.945495,...,4.792546,8.082185,5.309596,5.121320,3.259280,2.169233,2.006545,2.502644,1.784084,1.629458


In [35]:
result = result.reindex(
    sorted(
        result.columns,
        key=lambda x: (
            float(x.split('_')[0]),
            int(x.split('_')[-1])
        )
    ),
    axis=1
)

In [36]:
result

,57.0346_intensity_1,57.0346_intensity_2,57.0346_intensity_3,57.0346_intensity_4,57.0346_intensity_5,58.0296_intensity_1,58.0296_intensity_2,58.0296_intensity_3,58.0296_intensity_4,58.0296_intensity_5,...,985.6046_intensity_1,985.6046_intensity_2,985.6046_intensity_3,985.6046_intensity_4,985.6046_intensity_5,986.6105_intensity_1,986.6105_intensity_2,986.6105_intensity_3,986.6105_intensity_4,986.6105_intensity_5
cell_label_1,,,,,,,,,,,,,,,,,,,,,
0.0,0.722421,0.715801,0.839623,0.809700,0.810348,1.185498,1.183010,1.268906,1.248178,1.360782,...,1.185288,1.476967,1.649327,1.797921,1.840423,1.065894,1.352937,1.470545,1.571896,1.553651
1.0,0.822678,0.690730,0.798030,0.813259,0.894542,1.173516,1.101522,1.156898,1.126585,1.304175,...,1.069695,1.423279,1.571404,1.759689,1.780322,0.996263,1.267106,1.588876,1.665227,1.641532
3.0,0.697872,0.676140,0.826262,0.866502,0.941732,1.110374,1.199461,1.287048,1.276722,1.321521,...,1.115292,1.461638,1.640158,1.762730,1.858622,1.081938,1.305909,1.408406,1.616715,1.567569
4.0,0.693764,0.723285,0.781365,0.848504,0.791826,1.103839,1.058205,1.223086,1.197323,1.305049,...,1.208454,1.440774,1.560664,1.787655,1.798131,1.119037,1.284735,1.465121,1.597116,1.540583
5.0,0.890684,0.478694,0.618500,0.880955,0.696578,1.650921,0.907342,1.339190,0.718730,1.634189,...,0.833753,2.148722,2.206289,1.753013,2.271891,0.751409,1.677021,2.075442,1.677233,2.144233
6.0,0.833798,0.736912,0.826050,0.749442,0.943783,1.202757,1.187719,1.207694,1.358845,1.410376,...,1.079677,1.575801,1.586536,1.792664,1.910919,1.104250,1.361078,1.496755,1.557070,1.645229
7.0,0.677206,0.791876,0.843596,0.871436,0.800615,1.184140,1.175188,1.142650,1.247213,1.295629,...,1.121038,1.476227,1.632167,1.751537,1.849382,1.019787,1.392012,1.490248,1.562555,1.570597
8.0,0.754835,0.764326,0.826873,0.872043,0.877289,1.165626,1.128224,1.208492,1.175541,1.275105,...,1.056946,1.459993,1.666853,1.764073,1.890134,1.042329,1.247092,1.457048,1.579350,1.560787
11.0,0.698106,0.679427,0.852037,0.857635,0.846651,1.174150,1.190992,1.196493,1.252560,1.315341,...,1.147091,1.453081,1.604821,1.793256,1.784084,1.103378,1.348091,1.540075,1.467708,1.629458


In [37]:
result = result.T

In [38]:
result

cell_label_1,0.0,1.0,3.0,4.0,5.0,6.0,7.0,8.0,11.0,12.0,14.0,15.0,16.0,18.0,19.0
57.0346_intensity_1,0.722421,0.822678,0.697872,0.693764,0.890684,0.833798,0.677206,0.754835,0.698106,0.766876,0.702445,0.634265,0.861975,0.716357,0.646917
57.0346_intensity_2,0.715801,0.690730,0.676140,0.723285,0.478694,0.736912,0.791876,0.764326,0.679427,0.687371,0.655617,0.649049,0.766997,0.788847,0.633651
57.0346_intensity_3,0.839623,0.798030,0.826262,0.781365,0.618500,0.826050,0.843596,0.826873,0.852037,0.736315,0.796539,0.737792,0.763191,0.753901,0.771094
57.0346_intensity_4,0.809700,0.813259,0.866502,0.848504,0.880955,0.749442,0.871436,0.872043,0.857635,0.864059,0.889979,0.773557,0.753147,0.902093,0.753469
57.0346_intensity_5,0.810348,0.894542,0.941732,0.791826,0.696578,0.943783,0.800615,0.877289,0.846651,0.855521,0.855237,0.879108,0.923911,0.841814,0.909110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986.6105_intensity_1,1.065894,0.996263,1.081938,1.119037,0.751409,1.104250,1.019787,1.042329,1.103378,1.045760,1.122577,1.108778,0.978469,1.018829,0.972193
986.6105_intensity_2,1.352937,1.267106,1.305909,1.284735,1.677021,1.361078,1.392012,1.247092,1.348091,1.381946,1.330067,1.292429,1.396385,1.312526,1.278871
986.6105_intensity_3,1.470545,1.588876,1.408406,1.465121,2.075442,1.496755,1.490248,1.457048,1.540075,1.593760,1.476706,1.420004,1.317042,1.443657,1.397438
986.6105_intensity_4,1.571896,1.665227,1.616715,1.597116,1.677233,1.557070,1.562555,1.579350,1.467708,1.616241,1.523591,1.544800,1.553918,1.593210,1.558536


In [39]:
group_names = []

for i in range(0, len(result), 5):
    group_name = result.index[i].split('_')[0]
    group_names.append(group_name)

grouped_result = pd.DataFrame()

for i, group_name in enumerate(group_names):
    group_data = result.iloc[i*5:(i+1)*5]
    group_data.index = [group_name] * len(group_data)
    grouped_result = pd.concat([grouped_result, group_data])

print(grouped_result)

cell_label_1      0.0       1.0       3.0       4.0       5.0       6.0   \
57.0346       0.722421  0.822678  0.697872  0.693764  0.890684  0.833798   
57.0346       0.715801  0.690730  0.676140  0.723285  0.478694  0.736912   
57.0346       0.839623  0.798030  0.826262  0.781365  0.618500  0.826050   
57.0346       0.809700  0.813259  0.866502  0.848504  0.880955  0.749442   
57.0346       0.810348  0.894542  0.941732  0.791826  0.696578  0.943783   
...                ...       ...       ...       ...       ...       ...   
986.6105      1.065894  0.996263  1.081938  1.119037  0.751409  1.104250   
986.6105      1.352937  1.267106  1.305909  1.284735  1.677021  1.361078   
986.6105      1.470545  1.588876  1.408406  1.465121  2.075442  1.496755   
986.6105      1.571896  1.665227  1.616715  1.597116  1.677233  1.557070   
986.6105      1.553651  1.641532  1.567569  1.540583  2.144233  1.645229   

cell_label_1      7.0       8.0       11.0      12.0      14.0      15.0  \
57.0346    

In [40]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 假设 'grouped_result' 是之前的 DataFrame，按照索引列分组
x_values = np.array([0, 15, 30, 45, 60]).reshape(-1, 1)  # x 轴

# 初始化线性回归模型
model = LinearRegression()

# 创建一个空的列表，用来存储每个组的回归结果
regression_results = []

# 按索引分组
grouped = grouped_result.groupby(grouped_result.index)

# Step 1: 对每个组进行回归计算
for group_name, group_data in grouped:
    group_results = {}  # 存储每个组的回归结果
    
    # 对组内每列数据进行回归
    for col_name in group_data.columns:
        y_values = group_data[col_name].values  # y值
        
        # 进行线性回归
        model.fit(x_values, y_values)
        
        # 获取回归方程参数
        slope = model.coef_[0]
        intercept = model.intercept_
        r2 = model.score(x_values, y_values)  # 计算 R² 值
        
        # 将结果存储到字典中，列名格式为原列名加上后缀 '_*'
        group_results[f'{col_name}_Slope'] = slope
        group_results[f'{col_name}_Intercept'] = intercept
        group_results[f'{col_name}_R2'] = r2
    
    # 将每个组的回归结果存入列表
    regression_results.append(group_results)

# Step 2: 使用 pd.DataFrame() 将结果列表转换为 DataFrame
regression_df = pd.DataFrame(regression_results)

# 将回归结果的行名设置为原来的组名
regression_df.index = grouped_result.index.unique()

# 输出回归结果
print(regression_df)



          0.0_Slope  0.0_Intercept    0.0_R2  1.0_Slope  1.0_Intercept  \
57.0346    0.028065       0.902526  0.949659   0.024055       0.890765   
58.0296    0.076804      10.666251  0.981168   0.066142      11.131323   
59.0139    0.006424       1.020920  0.918126   0.008800       0.927548   
67.0191    0.020749       1.471746  0.954878   0.022234       1.206402   
71.0137    0.004156       0.354042  0.935401   0.004159       0.285396   
...             ...            ...       ...        ...            ...   
882.5853   0.010875       1.263741  0.930282   0.011718       1.169345   
921.5991   0.007963       1.164090  0.829850   0.011258       1.094069   
923.615   -0.038203      27.541038  0.799865  -0.037990      29.443375   
985.6046   0.007485       2.233078  0.937115   0.008745       2.177265   
986.6105   0.014410       4.698974  0.905418   0.017165       4.370162   

            1.0_R2  3.0_Slope  3.0_Intercept    3.0_R2  4.0_Slope  ...  \
57.0346   0.911190   0.024854       0

In [41]:
regression_df.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/MeanByCellLabel1LRResults10000.csv",index=False)

In [42]:
# 假设 'regression_df' 已经是计算好的回归结果

# Step 1: 提取所有包含 R2 的列
r2_columns = [col for col in regression_df.columns if '_R2' in col]

# Step 2: 创建一个字典来存储每个条件下符合的行数
r2_conditions = {0.9: 0, 0.8: 0, 0.7: 0}

# Step 3: 对每一行（代谢物）进行统计
for idx, row in regression_df.iterrows():
    # 计算当前代谢物符合各个条件的簇数
    r2_values = row[r2_columns]
    
    # 计算符合条件的簇数
    for threshold in r2_conditions.keys():
        # 统计 R2 大于等于 threshold 的细胞簇数量
        count = (r2_values >= threshold).sum()
        if count >= len(r2_values) / 2:
        # if count == len(r2_values):  
            r2_conditions[threshold] += 1

# Step 4: 输出结果
print(r2_conditions)

{0.9: 354, 0.8: 535, 0.7: 590}


In [ ]:
indexMergeSlice = indexMerge.iloc[5000:10000,:]
print(indexMergeSlice.tail())
for index, row in indexMergeSlice.iterrows():
    for i in range(5):
        SpotIndex = row[i]
        cellLabel = coordsLabelIntensity[coordsLabelIntensity.iloc[:, 0] == SpotIndex]['cell_label']
        indexMergeSlice.at[index, f'cell_label_{i+1}'] = cellLabel.iloc[0]

print(indexMergeSlice)
intensity_dict = coordsLabelIntensity.set_index(
    coordsLabelIntensity.columns[0]
).to_dict(orient='index')
columns_from_third = coordsLabelIntensity.columns[3:]

for index, row in indexMergeSlice.iterrows():
    for i in range(5):
        SpotIndex = row[i]
        if SpotIndex in intensity_dict:
            for col_name in columns_from_third:
                indexMergeSlice.at[
                    index,
                    f'{col_name}_intensity_{i+1}'
                ] = intensity_dict[SpotIndex].get(col_name, None)

print(indexMergeSlice)
indexMergeSlice.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabelIntensity10000.csv")
indexMergeSlice = pd.concat([indexMergeSlice.iloc[:,[5]],indexMergeSlice.iloc[:,10:]],axis=1)
grouped = indexMergeSlice.groupby(indexMergeSlice.columns[0])

print(indexMergeSlice.columns[0])

grouped_mean = grouped.mean()

result = pd.DataFrame(grouped_mean)
result = result.reindex(
    sorted(
        result.columns,
        key=lambda x: (
            float(x.split('_')[0]),
            int(x.split('_')[-1])
        )
    ),
    axis=1
)
result = result.T
group_names = []

for i in range(0, len(result), 5):
    group_name = result.index[i].split('_')[0]
    group_names.append(group_name)

grouped_result = pd.DataFrame()

for i, group_name in enumerate(group_names):
    group_data = result.iloc[i*5:(i+1)*5]
    group_data.index = [group_name] * len(group_data)
    grouped_result = pd.concat([grouped_result, group_data])

print(grouped_result)

# 假设 'grouped_result' 是之前的 DataFrame，按照索引列分组
x_values = np.array([0, 15, 30, 45, 60]).reshape(-1, 1)  # x 轴

# 初始化线性回归模型
model = LinearRegression()

# 创建一个空的列表，用来存储每个组的回归结果
regression_results = []

# 按索引分组
grouped = grouped_result.groupby(grouped_result.index)

# Step 1: 对每个组进行回归计算
for group_name, group_data in grouped:
    group_results = {}  # 存储每个组的回归结果
    
    # 对组内每列数据进行回归
    for col_name in group_data.columns:
        y_values = group_data[col_name].values  # y值
        
        # 进行线性回归
        model.fit(x_values, y_values)
        
        # 获取回归方程参数
        slope = model.coef_[0]
        intercept = model.intercept_
        r2 = model.score(x_values, y_values)  # 计算 R² 值
        
        # 将结果存储到字典中，列名格式为原列名加上后缀 '_*'
        group_results[f'{col_name}_Slope'] = slope
        group_results[f'{col_name}_Intercept'] = intercept
        group_results[f'{col_name}_R2'] = r2
    
    # 将每个组的回归结果存入列表
    regression_results.append(group_results)

# Step 2: 使用 pd.DataFrame() 将结果列表转换为 DataFrame
regression_df = pd.DataFrame(regression_results)

# 将回归结果的行名设置为原来的组名
regression_df.index = grouped_result.index.unique()

# 输出回归结果
print(regression_df)
regression_df.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/MeanByCellLabel1LRResults10000.csv",index=False)
# 假设 'regression_df' 已经是计算好的回归结果

# Step 1: 提取所有包含 R2 的列
r2_columns = [col for col in regression_df.columns if '_R2' in col]

# Step 2: 创建一个字典来存储每个条件下符合的行数
r2_conditions = {0.9: 0, 0.8: 0, 0.7: 0}

# Step 3: 对每一行（代谢物）进行统计
for idx, row in regression_df.iterrows():
    # 计算当前代谢物符合各个条件的簇数
    r2_values = row[r2_columns]
    
    # 计算符合条件的簇数
    for threshold in r2_conditions.keys():
        # 统计 R2 大于等于 threshold 的细胞簇数量
        count = (r2_values >= threshold).sum()
        if count >= len(r2_values) / 2:
        # if count == len(r2_values):  
            r2_conditions[threshold] += 1

# Step 4: 输出结果
print(r2_conditions)


In [ ]:
def process_indexMerge_slice_pipeline(
    indexMerge: pd.DataFrame,
    coordsLabelIntensity: pd.DataFrame,
    start: int,
    chunk_size: int = 5000,
    x_points=(0, 15, 30, 45, 60),
    out_aligned_csv: str = None,
    out_lr_csv: str = None,
    save_lr_index: bool = False,   # 你原来 index=False；如果你希望保存代谢物行名就改 True
    thresholds=(0.9, 0.8, 0.7),
    use_half_rule: bool = True,    # True: >=一半簇；False: 必须全部簇都满足
):
    """
    对 indexMerge 的一个分块（start ~ start+chunk_size）完整执行：
    1) 补 cell_label_1..5
    2) 补 *_intensity_1..5
    3) 列处理 + 按 cell_label_1 分组求均值
    4) 整理成每个 mz 对应 5 行（intensity_1..5）的 grouped_result
    5) 对每个 mz 在每个 cell_label_1 组上做线性回归，输出 slope/intercept/R2
    6) 统计 r2>=阈值 且满足“>=一半簇”条件的代谢物个数（或全部簇）
    """

    # -------- 0) 取分块 --------
    end = min(start + chunk_size, len(indexMerge))
    indexMergeSlice = indexMerge.iloc[start:end, :].copy()

    # -------- 1) 构建快速查表字典 --------
    spot_col = coordsLabelIntensity.columns[0]  # 一般是 SpotIndex
    # 每个 SpotIndex -> 这一行(字典)
    intensity_dict = coordsLabelIntensity.set_index(spot_col).to_dict(orient='index')
    # 每个 SpotIndex -> cell_label
    cell_label_series = coordsLabelIntensity.set_index(spot_col)['cell_label']
    cell_label_dict = cell_label_series.to_dict()

    # -------- 2) 补 cell_label_1..5 --------
    # 假设 indexMergeSlice 前5列就是 SpotIndex_0,15,30,45,60（按你之前代码 row[i]）
    for idx, row in indexMergeSlice.iterrows():
        for i in range(5):
            SpotIndex = row[i]
            indexMergeSlice.at[idx, f'cell_label_{i+1}'] = cell_label_dict.get(SpotIndex, None)

    # -------- 3) 补 *_intensity_1..5 --------
    # 你这里改成 columns[3:]：一般是跳过 Unnamed/sample/cell_label
    columns_from_third = coordsLabelIntensity.columns[3:]

    for idx, row in indexMergeSlice.iterrows():
        for i in range(5):
            SpotIndex = row[i]
            row_dict = intensity_dict.get(SpotIndex, None)
            if row_dict is None:
                # SpotIndex 没找到就全留 None
                continue
            for col_name in columns_from_third:
                indexMergeSlice.at[idx, f'{col_name}_intensity_{i+1}'] = row_dict.get(col_name, None)

    # -------- 4) 保存对齐后的 slice（可选）--------
    if out_aligned_csv is not None:
        indexMergeSlice.to_csv(out_aligned_csv, index=False)

    # -------- 5) 只保留你后续要 groupby 的列结构 --------
    # 你原来：保留第6列 + 第11列及以后
    # 这里完全照搬你的写法（注意：iloc按位置）
    indexMergeSlice2 = pd.concat([indexMergeSlice.iloc[:, [5]], indexMergeSlice.iloc[:, 10:]], axis=1)

    # -------- 6) 按第一列分组求均值 --------
    grouped = indexMergeSlice2.groupby(indexMergeSlice2.columns[0])
    grouped_mean = grouped.mean()
    result = pd.DataFrame(grouped_mean)

    # -------- 7) 重排列：按 mz + intensity序号 --------
    result = result.reindex(
        sorted(
            result.columns,
            key=lambda x: (float(x.split('_')[0]), int(x.split('_')[-1]))
        ),
        axis=1
    )

    # -------- 8) 转置 + 每5行一个 mz 分组 --------
    resultT = result.T

    group_names = []
    for i in range(0, len(resultT), 5):
        group_name = resultT.index[i].split('_')[0]
        group_names.append(group_name)

    grouped_result = pd.DataFrame()
    for i, group_name in enumerate(group_names):
        group_data = resultT.iloc[i*5:(i+1)*5].copy()
        group_data.index = [group_name] * len(group_data)
        grouped_result = pd.concat([grouped_result, group_data])

    # -------- 9) 线性回归（每个 mz 一组；对每个 cell_label_1 列都做）--------
    x_values = np.array(list(x_points)).reshape(-1, 1)
    model = LinearRegression()

    regression_results = []
    grouped_mz = grouped_result.groupby(grouped_result.index)

    for mz_name, group_data in grouped_mz:
        group_results = {}
        for col_name in group_data.columns:
            y_values = group_data[col_name].values
            model.fit(x_values, y_values)
            slope = float(model.coef_[0])
            intercept = float(model.intercept_)
            r2 = float(model.score(x_values, y_values))

            group_results[f'{col_name}_Slope'] = slope
            group_results[f'{col_name}_Intercept'] = intercept
            group_results[f'{col_name}_R2'] = r2

        regression_results.append(group_results)

    regression_df = pd.DataFrame(regression_results)
    regression_df.index = list(grouped_mz.groups.keys())  # mz 的名字（就是 grouped_result 的组名）

    # -------- 10) 保存回归结果（可选）--------
    if out_lr_csv is not None:
        regression_df.to_csv(out_lr_csv, index=save_lr_index)

    # -------- 11) 统计：R2 达标且满足“>=一半簇/全部簇”的代谢物个数 --------
    r2_columns = [c for c in regression_df.columns if c.endswith('_R2')]
    r2_conditions = {thr: 0 for thr in thresholds}

    for _, row in regression_df.iterrows():
        r2_values = row[r2_columns]
        for thr in thresholds:
            count = (r2_values >= thr).sum()
            if use_half_rule:
                if count >= len(r2_values) / 2:
                    r2_conditions[thr] += 1
            else:
                if count == len(r2_values):
                    r2_conditions[thr] += 1

    return {
        "start": start,
        "end": end,
        "indexMergeSlice": indexMergeSlice,          # 含 cell_label + intensity 的大表（可能很大）
        "indexMergeSlice_group_input": indexMergeSlice2,
        "grouped_mean_result": result,               # 分组均值（cell_label_1 行 × 强度列）
        "grouped_result_5rows_per_mz": grouped_result,  # 每个 mz 5行
        "regression_df": regression_df,
        "r2_conditions": r2_conditions
    }





In [ ]:
# ========== 你之前“从10000开始每5000行取一次”的循环，可以这样直接喂给函数 ==========
# chunk_size = 5000
# for start in range(10000, len(indexMerge), chunk_size):
#     out_aligned = f"/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/alignedIndexMergeCellLabelIntensity_{start}.csv"
#     out_lr = f"/p2/zulab/jtian/data/SA/05_CAST/output_ROICalculateByCluster/MeanByCellLabel1LRResults_{start}.csv"
#     out = process_indexMerge_slice_pipeline(
#         indexMerge=indexMerge,
#         coordsLabelIntensity=coordsLabelIntensity,
#         start=start,
#         chunk_size=chunk_size,
#         out_aligned_csv=out_aligned,
#         out_lr_csv=out_lr,
#         save_lr_index=False,   # 你原来是 index=False
#         thresholds=(0.9, 0.8, 0.7),
#         use_half_rule=True
#     )
#     print(start, out["r2_conditions"])

In [ ]:
#重新找点的对应关系，先保证一定在同一簇里，再去找欧氏距离最近的作为对应点，然后进行组织分割，同组织内算均值